# Using the run database to find out what runs to process

In [1]:
import pandas as pd
import numpy as np
import re

def build_config_row(group):
    kin_setting = group["Kinematics Setting"].iloc[0]
    is_production = bool(production_pattern.match(str(kin_setting).strip()))

    n_target = 1 if is_production else 4
    n = min(n_target, len(group))
    sampled = rng.choice(group["Run Number"].values, size=n, replace=False)
    sampled = np.sort(sampled).astype(int).tolist()

    return pd.Series({
        "Experiment": group["Experiment"].iloc[0],
        "Kinematics Setting": kin_setting,
        "Target": group["Target"].iloc[0],
        "sample_run_numbers": sampled,
    })


In [25]:
run_db = pd.read_csv("rcdb_draft.csv")
config_cols = ["Experiment", "Kinematics Setting", "Target"]

special_change_runs = [26085, 26088, 26132, 26148, 26155, 26165, 26161, 26190, 26271, 
                        26284, 26302, 26361, 26448, 26489, 26593, 26612, 26613, 26641, 
                        26652, 26769, 26724, 26784]  # See notes_for_some_runs.md

# Sort so cumsum respects run-number order within each config
run_db = run_db.sort_values(config_cols + ["Run Number"]).reset_index(drop=True)

# Mark the exact runs where a setting change was noted
run_db["is_change_point"] = run_db["Run Number"].isin(special_change_runs)

# Cumulative count of change points within each kinematic config
# -> increments every time a change point is hit, giving each "regime" its own id
run_db["setting_group"] = run_db.groupby(config_cols)["is_change_point"].cumsum()

# Runs with setting_group > 0 belong to a non-default (post-change) setting
run_db["flagged_changed_setting"] = run_db["setting_group"] > 0

# Unique integer label per (config_cols + setting_group) combination
run_db["Configuration"] = run_db.groupby(config_cols + ["setting_group"]).ngroup()

# Manual override: force these runs to share one Configuration,
# regardless of what the automatic setting_group logic produced
manual_groups = [
    [26148, 26149, 26150],
    [26151, 26152, 26153, 26154],
    [26155, 26156, 26157],
]
# manually sort out
run_db.loc[run_db.Configuration == 1, "Configuration"] = 0
for target_value, group in enumerate(manual_groups):
    mask = run_db["Run Number"].isin(group)
    run_db.loc[mask, "Configuration"] = target_value + 1

    
run_db = run_db.loc[ (run_db["Run Number"] != 26084) & (run_db["Run Status"]!="Junk") & (run_db["Run Status"]!="Short") & (run_db["Target"]!="Carbon Hole") & (run_db["Kinematics Setting"]!="Beam Checkout") & (run_db["Target"]!="Home"), :]
run_db.sort_values(by = 'Run Number', inplace = True)
run_db["Configuration"] = pd.factorize(run_db["Configuration"])[0]

# rng = np.random.default_rng(42)

# # Matches kinematics settings like "12a", "3b", "104e", etc.
# production_pattern = re.compile(r"^\d+[a-e]$")


# normalized_db = (
#     run_db.groupby("Configuration")
#     .apply(build_config_row, include_groups=False)
# )
# normalized_db.index.name = "Configuration"

# # Manual override for Configuration 52
# manual_sample_run_numbers = {
#     52: [26483, 26484, 26485, 26486, 26487, 26488],
# }

# for config_num, runs in manual_sample_run_numbers.items():
#     normalized_db.at[config_num, "sample_run_numbers"] = runs

# normalized_db.to_csv("calib_run_list_by_run_period.csv")
# all_runs = np.concatenate(normalized_db["sample_run_numbers"].to_numpy())
# np.savetxt("calib_run_list.csv", all_runs, fmt="%d", delimiter=",")

## Tutorial

In [5]:
run_db = pd.read_csv("rcdb_draft.csv")

In [15]:
run_db.loc[:, "Run Number"]

0       26066
1       26067
2       26068
3       26069
4       26070
        ...  
1009    27076
1010    27077
1011    27078
1012    27079
1013    27080
Name: Run Number, Length: 1014, dtype: int64

In [17]:
df = pd.DataFrame([{"run_number":  0}])
print(df.run_number)

0    0
Name: run_number, dtype: int64


In [16]:
run_db.keys()

Index(['Run Number', 'Run Status', 'Experiment', 'Kinematics Setting',
       'Target', 'Beam energy (GeV)', 'Beam current (uA)', 'HMS Angle (deg)',
       'HMS Angle Uncertainty (deg)', 'HMS Angle Log Entry',
       'HMS Momentum (GeV/c)', 'HMS Polarity', 'HMS Momentum Log Entry',
       'SHMS Angle (deg)', 'SHMS Angle Uncertainty (deg)',
       'SHMS Angle Log Entry', 'SHMS Momentum (GeV/c)', 'SHMS Polarity',
       'SHMS Momentum Log Entry', 'Comments'],
      dtype='object')

In [18]:
run_db

,Run Number,Run Status,Experiment,Kinematics Setting,Target,Beam energy (GeV),Beam current (uA),HMS Angle (deg),HMS Angle Uncertainty (deg),HMS Angle Log Entry,HMS Momentum (GeV/c),HMS Polarity,HMS Momentum Log Entry,SHMS Angle (deg),SHMS Angle Uncertainty (deg),SHMS Angle Log Entry,SHMS Momentum (GeV/c),SHMS Polarity,SHMS Momentum Log Entry,Comments
0,26066,Junk,N-Delta,Beam Checkout,Home,1.424427,1.0,14.505,0.000,https://logbooks.jlab.org/entry/4480689,-1.413,electron,https://logbooks.jlab.org/entry/4480721,14.5050,0.0050,https://logbooks.jlab.org/entry/4480689,-1.4130,electron,https://logbooks.jlab.org/entry/4480699,Beam Checkout Procedure
1,26067,Junk,N-Delta,Beam Checkout,Carbon Hole,1.424427,2.5,14.505,0.000,https://logbooks.jlab.org/entry/4480689,-1.413,electron,https://logbooks.jlab.org/entry/4480721,14.5050,0.0050,https://logbooks.jlab.org/entry/4480689,-1.4130,electron,https://logbooks.jlab.org/entry/4480699,Orbit Lock Issues
2,26068,Junk,N-Delta,Beam Checkout,Carbon Hole,1.424427,2.5,14.505,0.000,https://logbooks.jlab.org/entry/4480689,-1.413,electron,https://logbooks.jlab.org/entry/4480721,14.5050,0.0050,https://logbooks.jlab.org/entry/4480689,-1.4130,electron,https://logbooks.jlab.org/entry/4480699,Orbit Lock Issues
3,26069,Junk,N-Delta,Beam Checkout,Carbon Hole,1.424427,2.5,14.505,0.000,https://logbooks.jlab.org/entry/4480689,-1.413,electron,https://logbooks.jlab.org/entry/4480721,14.5050,0.0050,https://logbooks.jlab.org/entry/4480689,-1.4130,electron,https://logbooks.jlab.org/entry/4480699,Orbit Lock Issues
4,26070,Junk,N-Delta,Beam Checkout,Carbon Hole,1.424427,2.5,14.505,0.000,https://logbooks.jlab.org/entry/4480689,-1.413,electron,https://logbooks.jlab.org/entry/4480721,14.5050,0.0050,https://logbooks.jlab.org/entry/4480689,-1.4130,electron,https://logbooks.jlab.org/entry/4480699,Orbit Lock Issues
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1009,27076,Good,VCS2,7c,10 cm LH2,2.113200,25.0,20.229,0.001,https://logbooks.jlab.org/entry/4506411,0.701,proton,https://logbooks.jlab.org/entry/4506954,15.4675,0.0025,https://logbooks.jlab.org/entry/4501016,-1.6387,electron,https://logbooks.jlab.org/entry/4506954,NaN
1010,27077,Good,VCS2,7c,10 cm LH2,2.113200,25.0,20.229,0.001,https://logbooks.jlab.org/entry/4506411,0.701,proton,https://logbooks.jlab.org/entry/4506954,15.4675,0.0025,https://logbooks.jlab.org/entry/4501016,-1.6387,electron,https://logbooks.jlab.org/entry/4506954,NaN
1011,27078,Good,VCS2,7c,10 cm LH2,2.113200,25.0,20.229,0.001,https://logbooks.jlab.org/entry/4506411,0.701,proton,https://logbooks.jlab.org/entry/4506954,15.4675,0.0025,https://logbooks.jlab.org/entry/4501016,-1.6387,electron,https://logbooks.jlab.org/entry/4506954,NaN
1012,27079,Good,VCS2,7c,10 cm LH2,2.113200,25.0,20.229,0.001,https://logbooks.jlab.org/entry/4506411,0.701,proton,https://logbooks.jlab.org/entry/4506954,15.4675,0.0025,https://logbooks.jlab.org/entry/4501016,-1.6387,electron,https://logbooks.jlab.org/entry/4506954,NaN


In [19]:
display(run_db)

,Run Number,Run Status,Experiment,Kinematics Setting,Target,Beam energy (GeV),Beam current (uA),HMS Angle (deg),HMS Angle Uncertainty (deg),HMS Angle Log Entry,HMS Momentum (GeV/c),HMS Polarity,HMS Momentum Log Entry,SHMS Angle (deg),SHMS Angle Uncertainty (deg),SHMS Angle Log Entry,SHMS Momentum (GeV/c),SHMS Polarity,SHMS Momentum Log Entry,Comments
0,26066,Junk,N-Delta,Beam Checkout,Home,1.424427,1.0,14.505,0.000,https://logbooks.jlab.org/entry/4480689,-1.413,electron,https://logbooks.jlab.org/entry/4480721,14.5050,0.0050,https://logbooks.jlab.org/entry/4480689,-1.4130,electron,https://logbooks.jlab.org/entry/4480699,Beam Checkout Procedure
1,26067,Junk,N-Delta,Beam Checkout,Carbon Hole,1.424427,2.5,14.505,0.000,https://logbooks.jlab.org/entry/4480689,-1.413,electron,https://logbooks.jlab.org/entry/4480721,14.5050,0.0050,https://logbooks.jlab.org/entry/4480689,-1.4130,electron,https://logbooks.jlab.org/entry/4480699,Orbit Lock Issues
2,26068,Junk,N-Delta,Beam Checkout,Carbon Hole,1.424427,2.5,14.505,0.000,https://logbooks.jlab.org/entry/4480689,-1.413,electron,https://logbooks.jlab.org/entry/4480721,14.5050,0.0050,https://logbooks.jlab.org/entry/4480689,-1.4130,electron,https://logbooks.jlab.org/entry/4480699,Orbit Lock Issues
3,26069,Junk,N-Delta,Beam Checkout,Carbon Hole,1.424427,2.5,14.505,0.000,https://logbooks.jlab.org/entry/4480689,-1.413,electron,https://logbooks.jlab.org/entry/4480721,14.5050,0.0050,https://logbooks.jlab.org/entry/4480689,-1.4130,electron,https://logbooks.jlab.org/entry/4480699,Orbit Lock Issues
4,26070,Junk,N-Delta,Beam Checkout,Carbon Hole,1.424427,2.5,14.505,0.000,https://logbooks.jlab.org/entry/4480689,-1.413,electron,https://logbooks.jlab.org/entry/4480721,14.5050,0.0050,https://logbooks.jlab.org/entry/4480689,-1.4130,electron,https://logbooks.jlab.org/entry/4480699,Orbit Lock Issues
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1009,27076,Good,VCS2,7c,10 cm LH2,2.113200,25.0,20.229,0.001,https://logbooks.jlab.org/entry/4506411,0.701,proton,https://logbooks.jlab.org/entry/4506954,15.4675,0.0025,https://logbooks.jlab.org/entry/4501016,-1.6387,electron,https://logbooks.jlab.org/entry/4506954,NaN
1010,27077,Good,VCS2,7c,10 cm LH2,2.113200,25.0,20.229,0.001,https://logbooks.jlab.org/entry/4506411,0.701,proton,https://logbooks.jlab.org/entry/4506954,15.4675,0.0025,https://logbooks.jlab.org/entry/4501016,-1.6387,electron,https://logbooks.jlab.org/entry/4506954,NaN
1011,27078,Good,VCS2,7c,10 cm LH2,2.113200,25.0,20.229,0.001,https://logbooks.jlab.org/entry/4506411,0.701,proton,https://logbooks.jlab.org/entry/4506954,15.4675,0.0025,https://logbooks.jlab.org/entry/4501016,-1.6387,electron,https://logbooks.jlab.org/entry/4506954,NaN
1012,27079,Good,VCS2,7c,10 cm LH2,2.113200,25.0,20.229,0.001,https://logbooks.jlab.org/entry/4506411,0.701,proton,https://logbooks.jlab.org/entry/4506954,15.4675,0.0025,https://logbooks.jlab.org/entry/4501016,-1.6387,electron,https://logbooks.jlab.org/entry/4506954,NaN


In [35]:
run_db = pd.read_csv("rcdb_draft.csv")
config_cols = ["Experiment", "Kinematics Setting", "Target"]

special_change_runs = [26085, 26088, 26132, 26148, 26155, 26165, 26161, 26190, 26271, 
                        26284, 26302, 26361, 26448, 26489, 26593, 26612, 26613, 26641, 
                        26652, 26769, 26724, 26784]  # See notes_for_some_runs.md

# Sort so cumsum respects run-number order within each config
run_db = run_db.sort_values(config_cols + ["Run Number"]).reset_index(drop=True)

# Mark the exact runs where a setting change was noted
run_db["is_change_point"] = run_db["Run Number"].isin(special_change_runs)

# Cumulative count of change points within each kinematic config
# -> increments every time a change point is hit, giving each "regime" its own id
run_db["setting_group"] = run_db.groupby(config_cols)["is_change_point"].cumsum()

# Runs with setting_group > 0 belong to a non-default (post-change) setting
run_db["flagged_changed_setting"] = run_db["setting_group"] > 0

# Unique integer label per (config_cols + setting_group) combination
run_db["Configuration"] = run_db.groupby(config_cols + ["setting_group"]).ngroup()

# Manual override: force these runs to share one Configuration,
# regardless of what the automatic setting_group logic produced
manual_groups = [
    [26148, 26149, 26150],
    [26151, 26152, 26153, 26154],
    [26155, 26156, 26157],
]
# manually sort out
run_db.loc[run_db.Configuration == 1, "Configuration"] = 0
for target_value, group in enumerate(manual_groups):
    mask = run_db["Run Number"].isin(group)
    run_db.loc[mask, "Configuration"] = target_value + 1

run_db.sort_values(by = 'Run Number', inplace = True)
run_db["Configuration"] = pd.factorize(run_db["Configuration"])[0]

In [56]:
for Experiment in ["VCS2", "N-Delta"]:
  for Kinematics_Setting in run_db.loc[(run_db["Experiment"]==Experiment), "Kinematics Setting"].unique():
    if len(run_db.loc[(run_db["Experiment"]==Experiment) & (run_db["Kinematics Setting"] == Kinematics_Setting), "HMS Angle (deg)"].unique()) > 1:
           print(Experiment, Kinematics_Setting)

VCS2 Cosmics
VCS2 Beam Checkout
VCS2 3c
VCS2 4c
N-Delta Carbon Elastics
N-Delta HMS Carbon SIEVE
N-Delta 1H Elastics
N-Delta BCM Calibration
N-Delta Cosmics
N-Delta 3c
N-Delta 4c
N-Delta 5c


In [ ]:
1c 2c